# SHapley Additive exPlanations (SHAP)

SHapley Additive exPlanations (SHAP) is a method that explains **individual predictions** of any machine learning model by fairly **distributing the prediction’s output among the input features**. It leverages concepts from cooperative game theory—specifically Shapley values—to **quantify each feature’s contribution to the final prediction**. By considering all possible combinations of features, SHAP provides consistent and **locally accurate attributions** (here "locally" implies the explanation focuses on **the contribution of features for a specific individual prediction** or data point, rather than explaining the model’s behavior across the entire dataset), making model decisions **interpretable** and helping users understand how each feature influences the prediction for a specific sample. 

**SHAP Goal**

Mathematically, SHAP decomposes a prediction into additive contributions from each feature.  
For sample $i$:

$$
\text{Prediction}_i = \text{baseValue} + \sum_{j=1}^{M} \text{SHAP}_{ij}
$$

Where:

- $M$ is the number of features

- $\text{baseValue}$ is the **average prediction** across all training data  

- $\text{SHAP}_{ij}$ is the contribution of feature  $j$ for sample $i$


**To compute the base value for a classification problem**:

The formula below illustrates how to calculate the base value for class 1.

$$\
\text{baseValue} = \frac{1}{N} \sum_{i=1}^{N} P(y = 1 \mid \mathbf{x}_i)
\$$

Where:
- $N$ is number of training samples, and

- $x_i = [x_{i1}, x_{i2}, \ldots, x_{iM}]$ is $i^{th}$ sample features and $M$ is the number features.


**To compute the base value for a regression problem**:

$$\
\text{baseValue} = \frac{1}{N} \sum_{i=1}^{N} \hat{y}_i
\$$

Where:

- $N$ is number of training samples, and

- $\hat{y}_i$ is model prediction for sample $i$.

The code below builds a **decision tree classifier** using a dataset (*Grades.csv*) to predict whether a student passes based on several features. It visualizes the trained tree and prints its decision rules for **interpretability**. To explain how the model makes predictions, it uses SHAP (SHapley Additive exPlanations) to compute **feature attributions**. A SHAP explainer is created for the model, and summary plots (both beeswarm and bar) are generated to show **global feature importance**. Additionally, for a specific data point (row 12), the code computes and prints the SHAP values for class 1 (e.g., "Pass"), breaks down the prediction into base value and individual feature contributions, and visualizes this explanation using a SHAP waterfall plot.

**Decision Tree Classification** 

In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.tree import export_text

# dataset
df = pd.read_csv('Grades.csv')

# Features and target variable
X = df.drop(columns=['Pass'])
y = df['Pass']

# Train a decision tree classifier
dct = DecisionTreeClassifier(max_depth=5)
dct.fit(X, y)

# Plot the decision tree
plt.figure(figsize=(20, 15))
plot_tree(dct, filled=True, feature_names = X.columns,  class_names=['No', 'Yes'])
plt.show()

# Display the decision tree rules
tree_rules = export_text(dct, feature_names= X.columns)
print("Decision Tree Rules:\n", tree_rules)

**Global Feature Importance using SHAP**

In [ ]:
import shap

# class index (e.g., 0 for 'No', 1 for 'Yes')
class_idx = 1

# Create an explainer (use shap.Explainer)
explainer = shap.Explainer(dct)

# Compute SHAP values
shap_values = explainer(X)

# Convert feature names to NumPy array explicitly to avoid indexing issue
feature_names = X.columns.to_numpy()

# Plot SHAP summary (beeswarm)
shap.summary_plot(shap_values.values[:, :, class_idx], X.values, feature_names=feature_names)

# Plot SHAP summary (bar)
shap.summary_plot(shap_values.values[:, :, class_idx], X.values, feature_names=feature_names, plot_type="bar")

**NOTE:** If your SHAP summary plot (beeswarm or bar) shows less number features than what you started with, it usually means the excluded features have **near-zero SHAP values**. Also note that, by default, SHAP's summary_plot displays only the most important features unless you specify otherwise (`shap.summary_plot(shap_values.values, X.values, feature_names=feature_names, max_display=10)`). The following code snippet show mean SHAP values for each feature.

In [ ]:
import numpy as np
# Take absolute SHAP values for class 1 only
mean_shap = np.abs(shap_values.values[:, :, class_idx]).mean(axis=0)
# Print mean SHAP values for each feature
for name, val in zip(feature_names, mean_shap):
    print(f"{name}: {val:.4f}")

**SHAP Explanation for a Single Prediction**
The code below generates a SHAP explanation for a single prediction (row 12) for a specific class (e.g., class 1). It prints the base value, individual SHAP feature contributions, and then reconstructs the model prediction by summing them. Finally, it creates a SHAP Explanation object and visualizes the feature impacts with a waterfall plot, showing how each feature pushes the prediction above or below the base value.

In [ ]:
# One prediction explanation for class 1
row = 12
# class index (e.g., 0 for 'No', 1 for 'Yes')
class_idx = 1  

print("Base value:", shap_values.base_values[row][class_idx])
print("SHAP values:", shap_values.values[row][:, class_idx])
print("Model prediction:", shap_values.base_values[row][class_idx] + shap_values.values[row][:, class_idx].sum())

# Create a new explanation object for class 1 only
shap_single = shap.Explanation(
    values=shap_values.values[row][:, class_idx],
    base_values=shap_values.base_values[row][class_idx],
    data=shap_values.data[row],
    feature_names=shap_values.feature_names
)

# Waterfall plot for the chosen row and class
shap.plots.waterfall(shap_single)